# Model Alignment with DPO: training and generation

CS 5326 PA2, Ali Hasnain. This notebook runs the training pipeline on Kaggle (2 Tesla T4 GPUs). Each stage calls a
script in `scripts/`, and every stage writes its metrics, configs and outputs to `outputs/`.

| Stage | Task | GPU | Output |
|---|---|---|---|
| Environment, package versions, tests | Task 1 | none | test log |
| Data preparation: UltraChat 30k/1k, HH-RLHF 15k/1k, UltraFeedback 15k/1k | Section 6 | none | `outputs/data/` |
| Base-model generations and persona LoRA-DPO | Tasks 2 and 5 | GPU 1 | `generations/`, `persona_lora/` |
| SFT on Qwen2.5-0.5B | Task 3 | GPU 0 | `sft/` |
| SFT sanity check on 3 prompts | Task 6 | GPU 0 | printed below |
| DPO on HH-RLHF, then on UltraFeedback | Task 4 | policy GPU 0, reference GPU 1 | `hh_dpo/`, `ultrafeedback_dpo/` |
| SFT and DPO generations, blinded pair files | Task 6 | both | `generations/`, `eval/` |
| Plots, results tables, output bundles | Task 8 | none | `plots/`, `report/` |

Judging (Task 7) needs Kaggle model-proxy access and runs in `kaggle_judge.ipynb`.

In [1]:
# Configuration
REPO_URL = "https://github.com/alihasnaink/model-alignment.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/model-alignment"

SMOKE_TEST = False  # True: run every stage on tiny data
OVERLAP_GPU_WORK = True  # generation and persona LoRA on GPU 1 while GPU 0 trains
RESUME_FROM = None  # earlier outputs folder to continue from
# "lean" deletes model weights after their last use; "full" keeps them (about 12 GB).
OUTPUT_MODE = "full"

SEED = 42
BASE_MODEL = "Qwen/Qwen2.5-0.5B"
PERSONA_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
JUDGE_MODEL = "google/gemini-3.6-flash"

# Guide configuration (manual sections 9.2 and 10.4). Each run saves its values in run_config.json.
SFT_HP = dict(max_length=512, batch_size=2, gradient_accumulation=16, learning_rate=2e-5, warmup_ratio=0.03,
              weight_decay=0.1, max_grad_norm=1.0, epochs=1, eval_every=100)
DPO_HP = dict(max_length=512, batch_size=1, gradient_accumulation=32, learning_rate=1e-6, beta=0.1,
              optimizer="adamw", weight_decay=0.0, max_grad_norm=1.0, epochs=1, eval_every=100)
PERSONA_HP = dict(epochs=3, learning_rate=1e-4, batch_size=2, gradient_accumulation=4, beta=0.1,
                  lora_rank=16, lora_alpha=32, lora_dropout=0.05)
# Same decoding for every model: greedy with a small repetition penalty.
GEN_HP = dict(batch_size=16, max_new_tokens=512, repetition_penalty=1.1, no_repeat_ngram_size=0)

## Task 1: environment and tests

In [2]:
import os, subprocess, sys
from pathlib import Path

if not Path(REPO_DIR, "submission_adapter.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Using existing repository: {REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
repo = Path(REPO_DIR).resolve()
os.chdir(repo)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())

# Keep Kaggle's CUDA build of PyTorch (torch is not in requirements.txt). torchao conflicts with PEFT on
# older Kaggle images.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

Cloning into '/kaggle/working/model-alignment'...


Commit: e9e8915b1c4290292515f016fb78e159ab56c213
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.0 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], returncode=0)

In [3]:
import datasets, peft, torch, transformers, trl

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| CUDA", torch.version.cuda)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)
print("trl / peft  :", trl.__version__, "/", peft.__version__)
print("GPUs        :", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Enable a GPU accelerator."
if not SMOKE_TEST:
    assert torch.cuda.device_count() >= 2, "Select the GPU T4 x2 accelerator (policy on cuda:0, reference on cuda:1)."

# Public adapter tests and our own edge-case and end-to-end tests.
test_env = {**os.environ, "PYTHONPATH": str(repo), "CUDA_VISIBLE_DEVICES": "0"}  # 1 GPU: no DataParallel in the TRL test
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_public_adapter_smoke.py"], check=True, env=test_env)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/"], check=True, env=test_env)

python      : 3.13.15
torch       : 2.11.0+cu128 | CUDA 12.8
transformers: 4.57.6
datasets    : 4.8.5
trl / peft  : 0.29.1 / 0.20.0
GPUs        : ['Tesla T4', 'Tesla T4']
Thu Oct 15 11:52:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       3MiB /  153

CompletedProcess(args=['/usr/bin/python3', '-m', 'pytest', '-q', 'tests/'], returncode=0)

In [4]:
# Helpers
import json, shutil, threading, time, zipfile
from IPython.display import FileLink, Image, Markdown, display

OUT = repo / ("outputs_smoke" if SMOKE_TEST else "outputs")
DATA, GEN, EVAL, LOGS = OUT / "data", OUT / "generations", OUT / "eval", OUT / "logs"
for d in (DATA, GEN, EVAL, LOGS):
    d.mkdir(parents=True, exist_ok=True)
ALPACA_EVAL = repo / "eval_sets/alpaca_eval_300_seed42.json"
HH_EVAL = repo / "eval_sets/hh_rlhf_eval_300_seed42.json"
PERSONA_EVAL = repo / "eval_sets/persona_style_eval.json"

if RESUME_FROM and Path(RESUME_FROM).exists():
    # Copy only files we do not already have, so newer local work is never overwritten.
    for src in Path(RESUME_FROM).rglob("*"):
        dest = OUT / src.relative_to(RESUME_FROM)
        if src.is_file() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
    print("Resumed artifacts from", RESUME_FROM)


def flags(hp):
    return [x for key, value in hp.items() for x in (f"--{key}", value)]


def env_for(gpus):
    env = {**os.environ, "PYTHONPATH": str(repo), "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false",
           "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    if gpus is not None:
        env["CUDA_VISIBLE_DEVICES"] = gpus
    return env


def run(cmd, gpus=None, log_name=None):
    # Run a script in the foreground, streaming output to the cell and to outputs/logs/<log_name>.log.
    cmd = [str(c) for c in cmd]
    log_path = LOGS / f"{log_name or Path(cmd[1]).stem}.log"
    print(f"$ [{'GPU ' + gpus if gpus else 'all GPUs'}] {' '.join(cmd)}", flush=True)
    start = time.time()
    with log_path.open("a") as log, subprocess.Popen(cmd, cwd=repo, env=env_for(gpus), stdout=subprocess.PIPE,
                                                       stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}); see {log_path}")
    print(f"   finished in {(time.time() - start) / 60:.1f} min", flush=True)


class Background:
    # Runs a list of commands sequentially on one GPU in a thread; output goes to a log file only.
    def __init__(self, name, commands, gpus):
        self.name, self.commands, self.gpus, self.error = name, commands, gpus, None
        self.log = LOGS / f"background_{name}.log"
        if OVERLAP_GPU_WORK:
            self.thread = threading.Thread(target=self._work, daemon=True)
            self.thread.start()
            print(f"Started background track '{name}' on GPU {gpus} ({len(commands)} commands) -> {self.log}")
        else:
            self.thread = None
            for cmd in commands:
                run(cmd, gpus=None, log_name=f"background_{name}")

    def _work(self):
        try:
            with self.log.open("a") as log:
                for cmd in self.commands:
                    cmd = [str(c) for c in cmd]
                    log.write(f"\n$ {' '.join(cmd)}\n"); log.flush()
                    start = time.time()
                    code = subprocess.run(cmd, cwd=repo, env=env_for(self.gpus), stdout=log, stderr=subprocess.STDOUT).returncode
                    log.write(f"   exit {code} after {(time.time() - start) / 60:.1f} min\n"); log.flush()
                    if code != 0:
                        raise RuntimeError(f"{cmd[1]} exited with {code}")
        except Exception as error:
            self.error = error

    def wait(self):
        if self.thread is None:
            return
        print(f"Waiting for background track '{self.name}'...", flush=True)
        self.thread.join()
        print("".join(self.log.read_text().splitlines(keepends=True)[-25:]))
        if self.error:
            raise RuntimeError(f"background track '{self.name}' failed: {self.error}. Full log: {self.log}")
        print(f"Background track '{self.name}' finished OK.")


def done(stage_dir):
    return (Path(stage_dir) / "DONE").exists()


KEEP_IN_PRUNED = {"config.json", "generation_config.json", "checkpoint_info.json", "PRUNED.txt"}

def prune_weights(*checkpoint_dirs):
    # Lean mode: drop weights/tokenizer files but keep config + checkpoint_info.json as a record of what was saved.
    if OUTPUT_MODE != "lean":
        return
    for ckpt in map(Path, checkpoint_dirs):
        if not ckpt.exists() or (ckpt / "PRUNED.txt").exists():
            continue
        freed = 0
        for item in ckpt.iterdir():
            if item.name in KEEP_IN_PRUNED:
                continue
            freed += sum(f.stat().st_size for f in item.rglob("*") if f.is_file()) if item.is_dir() else item.stat().st_size
            shutil.rmtree(item) if item.is_dir() else item.unlink()
        (ckpt / "PRUNED.txt").write_text("Weights removed (OUTPUT_MODE='lean') after all generations using them finished.\n")
        print(f"pruned {ckpt.relative_to(OUT)}  (freed {freed / 1e9:.2f} GB)")


def has_weights(ckpt):
    return any(Path(ckpt).glob("*.safetensors"))


def gen_cmd(model, inputs, output, generator, dataset, extra=()):
    return ["python", "scripts/generate.py", "--model", model, "--inputs", inputs, "--output", output,
            "--generator", generator, "--dataset", dataset, *flags(GEN_HP), *SMOKE_GEN, *extra]


# Tiny data sizes for SMOKE_TEST; empty for the real run.
SMOKE_PREP = ["--train_size", 200, "--validation_size", 50] if SMOKE_TEST else []
SMOKE_SFT = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_DPO = ["--max_train_examples", 64, "--max_val_examples", 16, "--eval_every", 1] if SMOKE_TEST else []
SMOKE_GEN = ["--limit", 6, "--max_new_tokens", 64] if SMOKE_TEST else []
SMOKE_PERSONA = ["--max_steps", 4] if SMOKE_TEST else []
print("Outputs ->", OUT)

Outputs -> /kaggle/working/model-alignment/outputs


## Data preparation
* **SFT:** UltraChat-200k `train_sft`, first user turn and assistant reply of each conversation, 30,000 / 1,000 split.
* **HH-RLHF DPO:** single-turn pairs from `Anthropic/hh-rlhf`, 15,000 train and 1,000 validation.
* **UltraFeedback DPO:** `ultrafeedback_binarized`, 15,000 from `train_prefs` and 1,000 from `test_prefs`.
* Preference pairs are non-empty, chosen differs from rejected, no duplicates, and the formatted prompt is at most 256 tokens.
* The fixed evaluation sets are in `eval_sets/`. All sampling uses seed 42.

In [5]:
ULTRACHAT_DIR, HH_DIR, UF_DIR = DATA / "ultrachat", DATA / "hh", DATA / "ultrafeedback"
if not (ULTRACHAT_DIR / "sft_valid.jsonl").exists():
    run(["python", "scripts/prepare_ultrachat.py", "--output_dir", ULTRACHAT_DIR, "--seed", SEED, *SMOKE_PREP])
if not (HH_DIR / "hh_valid.jsonl").exists():
    run(["python", "scripts/prepare_preferences.py", "--track", "hh", "--output_dir", HH_DIR, "--seed", SEED, *SMOKE_PREP])
if not (UF_DIR / "ultrafeedback_valid.jsonl").exists():
    run(["python", "scripts/prepare_preferences.py", "--track", "ultrafeedback", "--output_dir", UF_DIR, "--seed", SEED, *SMOKE_PREP])
if not ALPACA_EVAL.exists():
    run(["python", "scripts/prepare_alpaca_eval.py", "--output", ALPACA_EVAL, "--num_prompts", 300, "--seed", SEED])

# Keep a copy of the exact eval subsets next to the results (manual: save the exact subsets used).
shutil.copytree(repo / "eval_sets", DATA / "eval_sets", dirs_exist_ok=True)
for path in sorted(DATA.rglob("*.json*")):
    if path.suffix == ".jsonl" or path.parent.name == "eval_sets":
        rows = sum(1 for _ in path.open()) if path.suffix == ".jsonl" else len(json.loads(path.read_text()))
        print(f"{rows:>7}  {path.relative_to(OUT)}")

$ [all GPUs] python scripts/prepare_ultrachat.py --output_dir /kaggle/working/model-alignment/outputs/data/ultrachat --seed 42

Generating train_sft split: 100%|██████████| 207865/207865 [00:04<00:00, 48363.00 examples/s]

Generating test_sft split: 100%|██████████| 23110/23110 [00:00<00:00, 56811.84 examples/s]

Generating train_gen split: 100%|██████████| 256032/256032 [00:03<00:00, 66599.21 examples/s]

Generating test_gen split: 100%|██████████| 28304/28304 [00:00<00:00, 71703.78 examples/s]

Filter: 100%|██████████| 207865/207865 [00:11<00:00, 17575.31 examples/s]

Map: 100%|██████████| 207843/207843 [00:25<00:00, 8139.68 examples/s]
Usable rows: 207843 / 207865
   finished in 1.2 min
$ [all GPUs] python scripts/prepare_preferences.py --track hh --output_dir /kaggle/working/model-alignment/outputs/data/hh --seed 42

Generating train split: 100%|██████████| 160800/160800 [00:02<00:00, 63821.26 examples/s]

Generating test split: 100%|██████████| 8552/8552 [00:00<00:00, 67263.90 exa

## Tasks 2, 3 and 5: baselines, persona LoRA-DPO and SFT
GPU 1 runs the base-model generations and the persona LoRA-DPO run while GPU 0 trains SFT.

In [6]:
PERSONA_OUT = OUT / "persona_lora"
track_a = [
    gen_cmd(BASE_MODEL, ALPACA_EVAL, GEN / "base_alpaca_eval.json", "qwen2.5-0.5b-base", "alpaca_eval"),
    gen_cmd(BASE_MODEL, HH_EVAL, GEN / "base_hh_eval.json", "qwen2.5-0.5b-base", "hh_rlhf_style_eval"),
    gen_cmd(PERSONA_MODEL, PERSONA_EVAL, GEN / "persona_base_persona_eval.json", "smollm2-360m-instruct",
            "persona_style_eval", ["--prompt_format", "chat"]),
]
if not done(PERSONA_OUT):
    track_a.append(["python", "scripts/train_persona_lora.py", "--model", PERSONA_MODEL, "--output_dir", PERSONA_OUT,
                    "--seed", SEED, *flags(PERSONA_HP), *SMOKE_PERSONA])
track_a.append(gen_cmd(PERSONA_MODEL, PERSONA_EVAL, GEN / "persona_lora_persona_eval.json", "smollm2-360m-instruct-lora-dpo",
                       "persona_style_eval", ["--prompt_format", "chat", "--adapter", PERSONA_OUT / "final"]))
bg_a = Background("baseline_and_persona", track_a, gpus="1" if torch.cuda.device_count() > 1 else "0")

Started background track 'baseline_and_persona' on GPU 1 (5 commands) -> /kaggle/working/model-alignment/outputs/logs/background_baseline_and_persona.log


In [7]:
SFT_OUT = OUT / "sft"
if done(SFT_OUT):
    print("SFT already done:", json.loads((SFT_OUT / "summary.json").read_text()))
else:
    run(["python", "scripts/train_sft.py", "--model", BASE_MODEL,
         "--train", ULTRACHAT_DIR / "sft_train.jsonl", "--validation", ULTRACHAT_DIR / "sft_valid.jsonl",
         "--output_dir", SFT_OUT, "--seed", SEED, *flags(SFT_HP), *SMOKE_SFT], gpus="0")
print(json.loads((SFT_OUT / "summary.json").read_text()))
prune_weights(SFT_OUT / "final")  # downstream stages only use SFT best

$ [GPU 0] python scripts/train_sft.py --model Qwen/Qwen2.5-0.5B --train /kaggle/working/model-alignment/outputs/data/ultrachat/sft_train.jsonl --validation /kaggle/working/model-alignment/outputs/data/ultrachat/sft_valid.jsonl --output_dir /kaggle/working/model-alignment/outputs/sft --seed 42 --max_length 512 --batch_size 2 --gradient_accumulation 16 --learning_rate 2e-05 --warmup_ratio 0.03 --weight_decay 0.1 --max_grad_norm 1.0 --epochs 1 --eval_every 100
Train examples: 30000 | micro-batches/epoch: 15000 | optimizer steps (total): 938
[step 0] val_loss=1.4192 (best=inf @ None)

optimizer steps:  11%| | 100/938 [08:28<1:11:38,  5.13s/it, best_val=n/a, epoch=1/1, loss=1.4466, lr=1.97e-05]
                                                                                                              
[step 100] val_loss=1.4519 (best=1.4519 @ 100)

optimizer steps:  11%| | 101/938 [09:25<1:53:45,  8.15s/it, best_val=n/a, epoch=1/1, loss=1.4466, lr=1.97e-05]
optimizer steps:  11%| | 101/93

In [8]:
bg_a.wait()
print(json.dumps(json.loads((PERSONA_OUT / "summary.json").read_text()), indent=2))

Waiting for background track 'baseline_and_persona'...
                                               
{'train_runtime': 195.073, 'train_samples_per_second': 3.076, 'train_steps_per_second': 0.384, 'train_loss': 0.16522161153455575, 'epoch': 3.0}

100%|██████████| 75/75 [03:15<00:00,  2.60s/it]

100%|██████████| 5/5 [00:00<00:00,  5.12it/s]
{
  "eval_loss": 0.01046593114733696,
  "eval_runtime": 1.2429,
  "eval_samples_per_second": 16.091,
  "eval_steps_per_second": 4.023
}
   exit 0 after 3.6 min

$ python scripts/generate.py --model HuggingFaceTB/SmolLM2-360M-Instruct --inputs /kaggle/working/model-alignment/eval_sets/persona_style_eval.json --output /kaggle/working/model-alignment/outputs/generations/persona_lora_persona_eval.json --generator smollm2-360m-instruct-lora-dpo --dataset persona_style_eval --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --prompt_format chat --adapter /kaggle/working/model-alignment/outputs/persona_lora/final
smollm2

## Task 6: SFT sanity check
Three SFT generations, checked by hand for the prompt format, EOS behaviour and length.

In [9]:
sanity = GEN / "sanity" / "sft_best_sanity.json"
if has_weights(SFT_OUT / "best"):
    run(gen_cmd(SFT_OUT / "best", ALPACA_EVAL, sanity, "sft-sanity", "alpaca_eval",
                ["--limit", 3, "--print_examples", 3, "--no-resume"]), gpus="0")
rows = json.loads(sanity.read_text())
for r in rows:
    print(f"--- {r['instruction'][:150]}\n>>> {r['output'][:600]}\n")
assert all(r["output"].strip() for r in rows), "SFT produced empty outputs - check prompt format / EOS handling"
print("token counts:", [r["num_new_tokens"] for r in rows], "(512 = hit the cap; mostly lower means EOS works)")

$ [GPU 0] python scripts/generate.py --model /kaggle/working/model-alignment/outputs/sft/best --inputs /kaggle/working/model-alignment/eval_sets/alpaca_eval_300_seed42.json --output /kaggle/working/model-alignment/outputs/generations/sanity/sft_best_sanity.json --generator sft-sanity --dataset alpaca_eval --batch_size 16 --max_new_tokens 512 --repetition_penalty 1.1 --no_repeat_ngram_size 0 --limit 3 --print_examples 3 --no-resume
sft-sanity on alpaca_eval: 3 prompts, 0 already done, 3 to generate
The tokenizer you are loading from '/kaggle/working/model-alignment/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.

--- [alpaca-eval-42-0001] Hi, my sister and her girlfriends want me to play kickball with them. Can you explain how the game is p

## Task 4: DPO on HH-RLHF and UltraFeedback
Both runs start from the SFT `best` checkpoint. The policy (fp32 weights, fp16 autocast) is on `cuda:0` and the frozen
fp16 reference on `cuda:1`. Beta 0.1, constant learning rate 1e-6, effective batch 32. The saved best checkpoint has the
highest validation reward accuracy. GPU 1 also generates the outputs that are already possible.

In [10]:
def run_dpo(name, data_dir, track):
    out = OUT / name
    if done(out):
        print(f"{name} already done:", json.loads((out / "summary.json").read_text()))
        return
    run(["python", "scripts/train_dpo.py", "--sft_checkpoint", SFT_OUT / "best",
         "--train", data_dir / f"{track}_train.jsonl", "--validation", data_dir / f"{track}_valid.jsonl",
         "--output_dir", out, "--run_name", name, "--seed", SEED, *flags(DPO_HP), *SMOKE_DPO])
    print(json.loads((out / "summary.json").read_text()))

HH_OUT, UF_OUT = OUT / "hh_dpo", OUT / "ultrafeedback_dpo"
bg_b = Background("sft_generations", [
    gen_cmd(SFT_OUT / "best", ALPACA_EVAL, GEN / "sft_alpaca_eval.json", "qwen2.5-0.5b-sft", "alpaca_eval"),
    gen_cmd(SFT_OUT / "best", HH_EVAL, GEN / "sft_hh_eval.json", "qwen2.5-0.5b-sft", "hh_rlhf_style_eval"),
], gpus="1" if torch.cuda.device_count() > 1 else "0")
run_dpo("hh_dpo", HH_DIR, "hh")
bg_b.wait()
prune_weights(HH_OUT / "final")  # only HH best is used for generation

Started background track 'sft_generations' on GPU 1 (2 commands) -> /kaggle/working/model-alignment/outputs/logs/background_sft_generations.log
$ [all GPUs] python scripts/train_dpo.py --sft_checkpoint /kaggle/working/model-alignment/outputs/sft/best --train /kaggle/working/model-alignment/outputs/data/hh/hh_train.jsonl --validation /kaggle/working/model-alignment/outputs/data/hh/hh_valid.jsonl --output_dir /kaggle/working/model-alignment/outputs/hh_dpo --run_name hh_dpo --seed 42 --max_length 512 --batch_size 1 --gradient_accumulation 32 --learning_rate 1e-06 --beta 0.1 --optimizer adamw --weight_decay 0.0 --max_grad_norm 1.0 --epochs 1 --eval_every 100
The tokenizer you are loading from '/kaggle/working/model-alignment/outputs/sft/best' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this

In [11]:
bg_c = Background("hh_dpo_generations", [
    gen_cmd(HH_OUT / "best", HH_EVAL, GEN / "hh_dpo_hh_eval.json", "qwen2.5-0.5b-hh-dpo", "hh_rlhf_style_eval"),
], gpus="1" if torch.cuda.device_count() > 1 else "0")
run_dpo("ultrafeedback_dpo", UF_DIR, "ultrafeedback")
bg_c.wait()
prune_weights(HH_OUT / "best", UF_OUT / "final")
run(gen_cmd(UF_OUT / "best", ALPACA_EVAL, GEN / "ultrafeedback_dpo_alpaca_eval.json", "qwen2.5-0.5b-ultrafeedback-dpo",
            "alpaca_eval"), gpus="0")
# Every generation is written now: both DPO runs and all SFT outputs no longer need the weights.
prune_weights(UF_OUT / "best", SFT_OUT / "best")
subprocess.run(["du", "-sh", str(OUT)], check=False)

Started background track 'hh_dpo_generations' on GPU 1 (1 commands) -> /kaggle/working/model-alignment/outputs/logs/background_hh_dpo_generations.log
$ [all GPUs] python scripts/train_dpo.py --sft_checkpoint /kaggle/working/model-alignment/outputs/sft/best --train /kaggle/working/model-alignment/outputs/data/ultrafeedback/ultrafeedback_train.jsonl --validation /kaggle/working/model-alignment/outputs/data/ultrafeedback/ultrafeedback_valid.jsonl --output_dir /kaggle/working/model-alignment/outputs/ultrafeedback_dpo --run_name ultrafeedback_dpo --seed 42 --max_length 512 --batch_size 1 --gradient_accumulation 32 --learning_rate 1e-06 --beta 0.1 --optimizer adamw --weight_decay 0.0 --max_grad_norm 1.0 --epochs 1 --eval_every 100
Traceback (most recent call last):
  File "/kaggle/working/model-alignment/scripts/train_dpo.py", line 174, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/model-alignment/scripts/train_dpo.py", line 70, in main
    train = load_json_records(args.train)[:

RuntimeError: command failed (1); see /kaggle/working/model-alignment/outputs/logs/train_dpo.log

## Task 6: blinded pairwise files
The A/B order is randomized per pair with seed 42. The model mapping is stored as hidden metadata and is never shown
to the judge.

In [ ]:
PAIRS = [
    ("hh_dpo_vs_sft", GEN / "sft_hh_eval.json", "sft", GEN / "hh_dpo_hh_eval.json", "hh_dpo"),
    ("ultrafeedback_dpo_vs_sft", GEN / "sft_alpaca_eval.json", "sft", GEN / "ultrafeedback_dpo_alpaca_eval.json", "ultrafeedback_dpo"),
    ("persona_lora_vs_base", GEN / "persona_base_persona_eval.json", "base_instruct", GEN / "persona_lora_persona_eval.json", "lora_dpo"),
    ("sft_vs_base_alpaca", GEN / "base_alpaca_eval.json", "base", GEN / "sft_alpaca_eval.json", "sft"),  # extra analysis
]
for stem, baseline, baseline_name, candidate, candidate_name in PAIRS:
    run(["python", "scripts/build_pairwise.py", "--baseline", baseline, "--baseline_name", baseline_name,
         "--candidate", candidate, "--candidate_name", candidate_name, "--output", EVAL / f"{stem}_pairs.jsonl", "--seed", SEED])

## Task 7: judging
Judging uses Kaggle model-proxy access and runs in `kaggle_judge.ipynb`. This cell only judges when the proxy is
available here, or with a mock judge in smoke-test mode.

In [ ]:
kaggle_judge_ready = "MODEL_PROXY_URL" in os.environ and "MODEL_PROXY_API_KEY" in os.environ
if kaggle_judge_ready:
    run(["python", "scripts/run_judging.py", "--eval_dir", EVAL, "--judge_model", JUDGE_MODEL, "--backend", "kaggle",
         "--batch_size", 5, "--sleep_seconds", 5, "--max_retries", 3])
elif SMOKE_TEST:
    run(["python", "scripts/run_judging.py", "--eval_dir", EVAL, "--backend", "mock", "--judge_model", "mock-longer-wins"])
else:
    print("No Kaggle model-proxy credentials in this kernel -> judging deferred to kaggle_judge.ipynb.")

## Task 8: plots, results tables and output bundles

In [ ]:
run(["python", "scripts/plot_metrics.py", "--outputs", OUT])
run(["python", "scripts/summarize_results.py", "--outputs", OUT])
for png in sorted((OUT / "plots").glob("*.png")):
    display(Markdown(f"**{png.name}**")); display(Image(filename=str(png)))

In [ ]:
WEIGHT_DIRS = {"best", "final", "trainer"}
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else repo.parent
suffix = "_smoke" if SMOKE_TEST else ""

def is_weight_file(rel):
    in_ckpt = any(part in WEIGHT_DIRS for part in rel.parts)
    lora_adapter = rel.parts[0] == "persona_lora" and "final" in rel.parts  # small (~35 MB) - keep it
    return in_ckpt and not lora_adapter and rel.name not in KEEP_IN_PRUNED

def is_raw_data(rel):
    return rel.parts[0] == "data" and rel.suffix == ".jsonl"  # regenerable training splits (~200 MB)

def make_zip(dest, keep):
    with zipfile.ZipFile(dest, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(OUT.rglob("*")):
            rel = path.relative_to(OUT)
            if path.is_file() and keep(rel):
                zf.write(path, Path("outputs") / rel)
    print(f"{dest.name:32s} {dest.stat().st_size / 1e6:8.1f} MB")
    return dest

# 1) Everything the report needs: metrics, run configs, summaries, plots, results tables, logs, data metadata,
#    generations, pairs/judgments, persona adapter. No base/SFT/DPO weights, no raw training splits.
report_zip = make_zip(WORK / f"report_bundle{suffix}.zip", lambda rel: not is_weight_file(rel) and not is_raw_data(rel))
# 2) Only what the judge notebook needs: generations, pair files and eval sets.
bench_zip = make_zip(WORK / f"benchmark_inputs{suffix}.zip",
                     lambda rel: rel.parts[0] in ("generations", "eval") or (rel.parts[0] == "data" and "eval_sets" in rel.parts))

if OUTPUT_MODE == "lean":
    for path in DATA.rglob("*.jsonl"):
        path.unlink()  # metadata.json files stay; the splits are reproducible from the prep scripts + seed
    print("lean mode: removed raw training splits from outputs/data (metadata kept)")

os.chdir(WORK)
display(Markdown("### Downloads (also in the Output tab after Save Version)"))
for zip_path in (report_zip, bench_zip):
    display(FileLink(zip_path.name))
os.chdir(repo)
subprocess.run(["du", "-sh", str(OUT), str(WORK)], check=False)

In [ ]:
summary = {name: json.loads((OUT / name / "summary.json").read_text()) for name in ("sft", "hh_dpo", "ultrafeedback_dpo", "persona_lora")
           if (OUT / name / "summary.json").exists()}
print(json.dumps(summary, indent=2, default=str))
judged = sorted(EVAL.glob("*_judged_summary.json"))
print("Judge summaries:", [p.name for p in judged] or "none yet - run kaggle_judge.ipynb")
display(Markdown((OUT / "report" / "results.md").read_text()[:6000]))
print("\nAll stages finished. Save Version now to preserve /kaggle/working (checkpoints + outputs).")